In [1]:
# ======================== 策略参数 ========================
hold_count       = 3      # 最大持仓数量
rebalance_days   = 5      # 调仓频率（交易日）
trend_period     = 25     # 趋势评分回归窗口
vol_period       = 20     # 波动率计算窗口

# ===================== 进攻性 ETF 资产池 =====================
OFFENSIVE_ETFS = [
    '511380.SH',   # 可转债ETF
    '513520.SH',   # 日经225ETF
    '159980.SZ',   # 有色金属ETF
    '160723.SZ',   # 华宝油气LOF（注意：LOF非纯ETF，存在折溢价风险）
    '159659.SZ',   # 纳指100ETF（招商）
    '159985.SZ',   # 豆粕ETF
    '513500.SH',   # 标普500ETF
    '518880.SH',   # 黄金ETF
    '513130.SH',   # 恒生科技ETF
    '510300.SH',   # 沪深300ETF
    '159783.SZ',   # 科创创业50ETF
    '159570.SZ',   # 港股通创新药ETF
]

# ===================== 依赖库 =====================
import pandas as pd
import numpy as np
import sys, os
# 定位本工程根（含 engine/）：data_source / smalltrader 在 engine/ 下，从 cwd 向上找根目录并入 sys.path
_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, 'engine')) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
sys.path.insert(0, os.path.join(_root, 'engine'))
sys.path.insert(0, _root)
import smalltrader as bigtrader    # 本地回测引擎（API 对齐平台 bigtrader）
import queries      # 本地 DuckDB：cn_fund_bar1d（后复权）离线读 data/ 下 parquet
import data_source  # 注册自动取数：bigtrader.run(data=缺省) 时按 market 从本地 DuckDB 拉全市场基金行情/分红折算/名称（对齐平台原版隐式取数）

# ===================== 因子计算 =====================

def calc_trend_score(close, period=25):
    """
    趋势评分 = 年化收益率 × R²
    score > 0 表示上行趋势，越大越强
    """
    n = len(close)
    if n < period:
        return pd.Series(np.nan, index=close.index, dtype=float)

    y = np.log(close.values)
    windows = np.lib.stride_tricks.sliding_window_view(y, window_shape=period)
    x = np.arange(period)

    sum_x  = x.sum()
    sum_x2 = (x ** 2).sum()
    denom  = period * sum_x2 - sum_x ** 2

    sum_y  = windows.sum(axis=1)
    sum_xy = (windows * x).sum(axis=1)

    slope      = (period * sum_xy - sum_x * sum_y) / denom
    intercept  = (sum_y - slope * sum_x) / period
    ann_return = np.exp(slope * 252) - 1

    y_pred    = slope[:, None] * x + intercept[:, None]
    residuals = windows - y_pred
    ss_res    = (residuals ** 2).sum(axis=1)
    sum_y2    = (windows ** 2).sum(axis=1)
    ss_tot    = sum_y2 - (sum_y ** 2) / period
    r2        = np.where(ss_tot == 0, 0.0, 1 - ss_res / ss_tot)

    score = ann_return * r2
    result = pd.Series(np.nan, index=close.index, dtype=float)
    result.iloc[period - 1:] = score
    return result


def calc_volatility(close, period=20):
    """滚动年化波动率（对数收益率标准差 × sqrt(252)）"""
    log_ret = np.log(close).diff()
    return log_ret.rolling(window=period).std() * np.sqrt(252)


# ===================== 初始化 =====================

def initialize(context: bigtrader.IContext):
    context.set_commission(bigtrader.PerOrder(buy_cost=0.0002, sell_cost=0.0002, min_cost=5))

    context.stock_num       = hold_count
    context.rebalance_days  = rebalance_days    

    if 'counter' not in context.user_store:
        context.user_store['counter'] = rebalance_days - 1

    # 因子行情：本地 DuckDB cn_fund_bar1d（后复权）
    con = queries.connect()
    q_start = context.add_trading_days(context.start_date, -200).strftime('%Y-%m-%d')
    q_end = context.end_date
    pool_sql = ', '.join("'%s'" % i for i in OFFENSIVE_ETFS)
    df = con.execute(
        "SELECT date, instrument, close FROM cn_fund_bar1d "
        "WHERE instrument IN (%s) AND date >= DATE '%s' AND date <= DATE '%s'"
        % (pool_sql, q_start, q_end)
    ).df()
    df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')   # 字符串日期，handle_data 用 == 比较

    df = df[df['instrument'].isin(OFFENSIVE_ETFS)].sort_values(['instrument', 'date'])
    df['score'] = df.groupby('instrument')['close'].transform(
        lambda x: calc_trend_score(x, period=trend_period)
    )
    df['volatility'] = df.groupby('instrument')['close'].transform(
        lambda x: calc_volatility(x, period=vol_period)
    )

    context.data = df


# ===================== 每日开盘前 =====================

def before_trading_start(context, data):
    pass


# ===================== 核心交易逻辑 =====================

def handle_data(context, data):
    current_date = data.current_dt.strftime("%Y-%m-%d")

    today = context.data[context.data['date'] == current_date]
    if today.empty:
        return

    context.user_store['counter'] += 1
    if context.user_store['counter'] != context.rebalance_days:
        return
    context.user_store['counter'] = 0

    current_hold = set(context.get_account_positions().keys())

    # ── 筛选进攻性标的 ────────────────────────────────────
    valid = today[
        today['score'].notna() &
        today['volatility'].notna()
    ].sort_values('score', ascending=False)

    selected      = valid.head(context.stock_num)
    stock_targets = selected['instrument'].tolist()

    # ── 无合格标的 → 清仓 ──────────────────────────────────
    if not stock_targets:
        for ins in current_hold:
            context.order_target_percent(ins, 0)
        return

    # ── 等权分配 ───────────────────────────────────────────
    target_set = set(stock_targets)
    w = 0.9 / max(len(stock_targets),1.5)  # 若目标持仓 <2 则按 1.5 等权分配，避免过度集中

    # 先卖出不在目标中的持仓
    for ins in current_hold - target_set:
        context.order_target_percent(ins, 0)

    # 按等权买入，若目标持仓与现有持仓相差 <3% 则跳过
    positions = context.get_account_positions()
    portfolio_value = context.portfolio.portfolio_value
    for ins in stock_targets:
        pos = positions.get(ins)
        if pos is not None:
            cur_w = pos.market_value / portfolio_value
            if abs(cur_w - w) < 0.03:
                continue
        context.order_target_percent(ins, w)

performance = bigtrader.run(
    market                 = bigtrader.Market.CN_FUND,
    frequency              = bigtrader.Frequency.DAILY,
    start_date             = '2020-01-01',
    end_date               = '2026-10-30',
    capital_base           = 1000000,
    initialize             = initialize,
    handle_data            = handle_data,
    before_trading_start   = before_trading_start,
    order_price_field_buy  = 'open',
    order_price_field_sell = 'open',
    render = "report"
)

#performance.render()


# %%

交易日历剔除无有效行情的占位日 3 个（疑似脏行/非交易日）: 2019-12-28, 2020-08-01, 2022-06-03


## BigTrader 回测报告

### 绩效摘要

| 指标    | 值 |
|-------|------|
| 累计收益率 | 291.9% |
| 年化收益率 | 23.15% |
| 基准收益率 | -4.02% |
| 阿尔法   | 0.23 |
| 贝塔    | 0.4 |
| 夏普比率  | 1.06 |
| 胜率    | 53.31% |
| 盈亏比   | 1.77 |
| 收益波动率 | 18.53% |
| 信息比率  | 1.05 |
| 最大回撤  | 17.04% |

### 收益走势

| 日期 | 策略收益率 | 基准收益率 | 相对收益率 |
|------|----------|----------|----------|
| 2020-01-02 | 0.0% | 0.0% | 0.0% |
| 2020-01-23 | -2.36% | -3.57% | 1.04% |
| 2020-02-28 | -5.27% | -5.11% | -1.09% |
| 2020-03-31 | -6.21% | -11.22% | 4.1% |
| 2020-04-30 | -3.75% | -5.77% | 0.54% |
| 2020-05-29 | -1.64% | -6.87% | 3.9% |
| 2020-06-30 | 0.09% | 0.28% | -1.84% |
| 2020-07-31 | 4.9% | 13.07% | -9.29% |
| 2020-08-31 | 8.08% | 15.99% | -9.07% |
| 2020-09-30 | 4.78% | 10.48% | -7.51% |
| 2020-10-30 | 4.25% | 13.08% | -10.21% |
| 2020-11-30 | 13.52% | 19.46% | -7.54% |
| 2020-12-31 | 16.6% | 25.51% | -9.76% |
| 2021-01-29 | 16.52% | 28.89% | -12.45% |
| 2021-02-26 | 23.38% | 28.53% | -7.21% |
| 2021-03-31 | 23.31% | 21.58% | -2.43% |
| 2

In [ ]:
#performance.render()